<a href="https://colab.research.google.com/github/DecioVdA/ProveComm/blob/main/Modelli/RandomForest_RidDispo_Fasce15min.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### Random Forest_RiduzioneDisponibilità

In questo esempio, al posto di analizzare solamente il totale giornaliero della targert feature.
Eliminerò quindi la variabile TransitiFRA, moltiplicherò il DataFrame per 96 valori gionralieri e concatenero' i dati Transiti su scaglioni di 15 minuti.

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import plotly.express as px
import requests
from io import BytesIO
import random
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

rd_seed = 42  # fissiamo un random seed per riproducibilità e replicabilità
np.random.seed(rd_seed)
random.seed(rd_seed)

rd_seed = 42

In [2]:
url = "https://github.com/DecioVdA/ProveComm/raw/refs/heads/main/DataSetAddestramento.xlsx"
response = requests.get(url)
df = pd.read_excel(BytesIO(response.content))

In [3]:
df.head()

,Data,GiornoSettimana,Scolastiche_VDA,Scolastiche_FRA,Scolastiche_CH,Festivo_ITA,Festivo_FRA,Festivo_CH,Dispo_TU,Dispo_Fre,TransitiFRA
0,2024-01-01,Lunedi,1,ABC,nullo,FE,FE,FE,1.0,1.0,2274
1,2024-01-02,Martedi,1,ABC,nullo,nullo,nullo,nullo,1.0,1.0,3160
2,2024-01-03,Mercoledi,1,ABC,nullo,nullo,nullo,nullo,1.0,1.0,2667
3,2024-01-04,Giovedi,1,ABC,nullo,nullo,nullo,nullo,1.0,1.0,2633
4,2024-01-05,Venerdi,1,ABC,nullo,nullo,nullo,nullo,1.0,1.0,2608


In [4]:
df = df.drop(['TransitiFRA'], axis= 1)
df.head()

,Data,GiornoSettimana,Scolastiche_VDA,Scolastiche_FRA,Scolastiche_CH,Festivo_ITA,Festivo_FRA,Festivo_CH,Dispo_TU,Dispo_Fre
0,2024-01-01,Lunedi,1,ABC,nullo,FE,FE,FE,1.0,1.0
1,2024-01-02,Martedi,1,ABC,nullo,nullo,nullo,nullo,1.0,1.0
2,2024-01-03,Mercoledi,1,ABC,nullo,nullo,nullo,nullo,1.0,1.0
3,2024-01-04,Giovedi,1,ABC,nullo,nullo,nullo,nullo,1.0,1.0
4,2024-01-05,Venerdi,1,ABC,nullo,nullo,nullo,nullo,1.0,1.0


Espansione del dataframe per aggiungere 96 valori giornalieri.

In [5]:
def espansione_dataFrame(d_frame):
  row_espanse = []
  for idx, row in df.iterrows():
      for fascia in range(1, 97):  # da 1 a 96
          new_row = row.copy()
          new_row["FasciaOraria"] = fascia
          row_espanse.append(new_row)
  df_espanso = pd.DataFrame(row_espanse)


df = espansione_dataFrame(df)


In [7]:
df